# KNN Implementation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from collections import Counter

class KNN_self:
    def __init__(self, k=3, task="classification", weighted=False):
        self.k = k
        self.task = task
        self.weighted = weighted
    def fit(self,X,y):
        self.X_train=X
        self.y_train=y
    def _euclidean_distance(self,x1,x2):
        ans = np.sqrt(np.sum((x1-x2)**2))
        return ans
    def predict(self, X):
        predictions = []
        for x in X:
            distances=[
                self._euclidean_distance(x,x_train)
                for x_train in self.X_train
            ]
            k_indices = np.argsort(distances)[:self.k]
            k_labels = np.array([self.y_train[i] for i in k_indices])
            if self.task == "classification":
                if self.weighted:
                    # weighted voting
                    weights = 1 / (np.array(distances)[k_indices] + 1e-5)
                    vote = {}
                    for label, weight in zip(k_labels, weights):
                        vote[label] = vote.get(label, 0) + weight
                    prediction = max(vote, key=vote.get)
                else:
                    # simple majority vote
                    prediction = Counter(k_labels).most_common(1)[0][0]

            else:  # regression
                if self.weighted:
                    weights = 1 / (np.array(distances)[k_indices] + 1e-5)
                    prediction = np.sum(weights * k_labels) / np.sum(weights)
                else:
                    prediction = np.mean(k_labels)

            predictions.append(prediction)
        return np.array(predictions)



---

# `sklearn.neighbors.KNeighborsClassifier`

```python
KNeighborsClassifier(
    n_neighbors=5,
    weights='uniform',
    algorithm='auto',
    leaf_size=30,
    p=2,
    metric='minkowski',
    metric_params=None,
    n_jobs=None
)
```

We’ll go **top → bottom**, in the order you should think about them.

---

## 1. `n_neighbors` (K) ⭐⭐⭐⭐⭐

```python
n_neighbors=5
```

### What it does

Number of neighbors used to make the prediction.

### Practical effects

* **Small K**

  * Complex decision boundaries
  * Sensitive to noise
* **Large K**

  * Smooth boundaries
  * Higher bias

### How to choose

* Start with: `sqrt(N)`
* Tune using **cross-validation**
* Always test **odd K** for binary classification

```python
n_neighbors=7
```

---

## 2. `weights` ⭐⭐⭐⭐

```python
weights='uniform'
```

### Options

* `'uniform'` → all neighbors count equally
* `'distance'` → closer neighbors count more
* callable → custom weighting function

### When to use what

| Scenario                   | Use        |
| -------------------------- | ---------- |
| Clean, well-separated data | `uniform`  |
| Noisy or uneven density    | `distance` |

```python
weights='distance'
```

This often gives a **free accuracy boost**.

---

## 3. `metric` ⭐⭐⭐⭐⭐

```python
metric='minkowski'
```

This defines **how distance is computed**.

### Common choices

| Metric        | When to use               |
| ------------- | ------------------------- |
| `'euclidean'` | Default numeric data      |
| `'manhattan'` | High-dimensional / robust |
| `'cosine'`    | Text / embeddings         |
| `'chebyshev'` | Max difference            |
| `'hamming'`   | Binary features           |

```python
metric='euclidean'
```

⚠️ Always scale data first (StandardScaler / MinMaxScaler).

---

## 4. `p` (only if metric = minkowski)

```python
p=2
```

### Meaning

Controls the Minkowski distance:

* `p=1` → Manhattan
* `p=2` → Euclidean

```python
metric='minkowski', p=1
```

Most of the time:
👉 Just use `metric='euclidean'` or `'manhattan'` directly.

---

## 5. `algorithm` ⭐⭐⭐

```python
algorithm='auto'
```

Controls **how neighbors are searched**.

### Options

| Algorithm     | Use case             |
| ------------- | -------------------- |
| `'auto'`      | Let sklearn decide   |
| `'kd_tree'`   | Low-dimensional data |
| `'ball_tree'` | Medium dimensions    |
| `'brute'`     | High-dim / cosine    |

```python
algorithm='ball_tree'
```

⚠️ `cosine` **requires** `brute`.

---

## 6. `leaf_size` ⭐⭐

```python
leaf_size=30
```

### What it does

Controls tree structure (KDTree / BallTree).

### Practical notes

* Smaller → faster queries, more memory
* Larger → slower queries, less memory
* Usually **don’t touch this**

Tune only if:

* Very large dataset
* Performance-critical application

---

## 7. `metric_params` ⭐

```python
metric_params=None
```

Extra parameters for **custom metrics**.

Example:

```python
metric='minkowski'
metric_params={'p': 3}
```

Most users never need this.

---

## 8. `n_jobs` ⭐⭐

```python
n_jobs=None
```

### What it does

Parallelizes distance computations.

```python
n_jobs=-1  # use all cores
```

Helpful when:

* Dataset is large
* Using brute-force search

---

# Full Practical Example (Best Practice)

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors=7,
    weights='distance',
    metric='manhattan',
    algorithm='auto',
    n_jobs=-1
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', knn)
])
```

---

# KNN Regression Differences

```python
from sklearn.neighbors import KNeighborsRegressor
```

Same parameters, but:

* Prediction = **mean** of neighbors
* `weights='distance'` is even more important

---

# What to Tune First (Priority Order)

1. **`n_neighbors`**
2. **`weights`**
3. **`metric`**
4. **`algorithm`**
5. `leaf_size` (rare)

---

# Common Mistakes 🚨

❌ Not scaling features
❌ Using cosine with KDTree
❌ Large K on small datasets
❌ Ignoring class imbalance
❌ Using KNN on very high-dimensional data

---

## Mental Model

> KNN is not “simple” — it’s **distance engineering + voting strategy**.



## Classification

In [ ]:
from sklearn.datasets import make_classification
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
X,y = make_classification(
    n_samples= 10000,
    n_features= 4,
    n_redundant=1,
    n_classes=2,
    random_state=67
)

In [ ]:
model_self = KNN_self(weighted=True, k=5)
model_original = KNeighborsClassifier(
    n_jobs=-1,
    n_neighbors=5,
    p=2,
    algorithm='auto'
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


pipe_self = Pipeline([
    ('scaler', StandardScaler()),
    ('self_model', model_self)
])

pipe_original = Pipeline([
    ('scaler', StandardScaler()),
    ('self_model', model_original)
])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
pipe_self.fit(X_train, y_train)

In [ ]:
pipe_original.fit(X_train, y_train)

In [ ]:
y_pred=pipe_self.predict(X_test)

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


# slow as this one does np based distance calculation whereas the sklearn does
# direct C based calculations which is much much faster

In [ ]:
y_pred=pipe_original.predict(X_test)

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "knn__n_neighbors": randint(1, 50),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
    "knn__algorithm": ["auto", "kd_tree", "ball_tree", "brute"]
}


In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])


In [ ]:
random_search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=50,          # number of random configs
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

In [ ]:
print(random_search.best_params_)
print(random_search.best_score_)

In [ ]:
y_pred=random_search.predict(X_test)

from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Regression

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
from sklearn.datasets import make_regression

X, y = make_regression(
    n_samples=10000,
    n_features=4,
    n_informative=3,
    noise=0.1,
    random_state=67
)


In [ ]:
model_self = KNN_self(weighted=True, k=5, task="pew")
model_original = KNeighborsRegressor(
    n_jobs=-1,
    n_neighbors=5,
    p=2,
    algorithm='auto'
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


pipe_self = Pipeline([
    ('scaler', StandardScaler()),
    ('self_model', model_self)
])

pipe_original = Pipeline([
    ('scaler', StandardScaler()),
    ('self_model', model_original)
])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
pipe_self.fit(X_train, y_train)

In [ ]:
pipe_original.fit(X_train, y_train)

In [ ]:
y_pred=pipe_self.predict(X_test)

from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))
print(mean_squared_error(y_test,y_pred))


# slow as this one does np based distance calculation whereas the sklearn does
# direct C based calculations which is much much faster

In [ ]:
y_pred=pipe_original.predict(X_test)

from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))
print(mean_squared_error(y_test,y_pred))


In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "knn__n_neighbors": randint(1, 50),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"],
    "knn__algorithm": ["auto", "kd_tree", "ball_tree", "brute"]
}


In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor())
])


In [ ]:
random_search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=50,          # number of random configs
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

In [ ]:
print(random_search.best_params_)
print(random_search.best_score_)

In [ ]:
y_pred=random_search.predict(X_test)

print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))
print(mean_squared_error(y_test,y_pred))